In [2]:
#  Imports & ROOT setup
import sys
from pathlib import Path
import warnings
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.ganblr.models import GANBLR  

import warnings
import logging

# Hide Keras warnings
warnings.filterwarnings("ignore", message="Do not pass an `input_shape`")
warnings.filterwarnings("ignore", message="Do not pass an `input_dim`")

# Hide pgmpy probability warnings
warnings.filterwarnings("ignore", message="Probability values don't exactly sum to 1", module="pgmpy")
logging.getLogger("pgmpy").setLevel(logging.ERROR)

# Hide sklearn teacher warnings (optional)
warnings.filterwarnings("ignore", message="X does not have valid feature names")


class CustomGANBLR(GANBLR):
   
    def train(self, dataset, size_category='small', *args, **kwargs):
        # Decide epochs and k based on dataset size
        if size_category == 'small':
            epochs = 100
            k = 2      # car
        elif size_category == 'medium':
            epochs = 100
            k = 1      # nursery, magic
        else:  # 'large'
            epochs = 120
            k = 1      # adult, shuttle

        model_name = "ganblr"
        dataset_name = dataset
        data_dir = f"{dataset_name}"

        # Honor explicit synthetic_dir if provided (pipeline passes this)
        explicit_synth_dir = kwargs.get('synthetic_dir')
        if explicit_synth_dir and isinstance(explicit_synth_dir, str):
            save_dir = explicit_synth_dir
        else:
            save_dir = os.path.join("synthetic", dataset_name, model_name)
        os.makedirs(save_dir, exist_ok=True)

        x_train_path = os.path.join(data_dir, "x_train.csv")
        y_train_path = os.path.join(data_dir, "y_train.csv")

        # Seeds for reproducibility
        seed = 42
        np.random.seed(seed)
        random.seed(seed)

        # Load training data
        X = pd.read_csv(x_train_path)          # has real feature names (e.g. fAlpha, fAsym, ...)
        y_df = pd.read_csv(y_train_path)       # keep as DataFrame to grab column name
        y = y_df.values.ravel()
        y_col = y_df.columns[0]                # e.g. "class" or "target"

        print(f"Loaded X shape: {X.shape}, y shape: {y.shape}")
        print(f"Feature columns: {list(X.columns)}")
        print(f"Label column   : {y_col}")

        # Use size-dependent k
        self.fit(X, y, k=k, epochs=epochs, batch_size=64)

        # Generate synthetic data and save
        syn_data = self.sample(X.shape[0])  # returns numpy array [X | y]

        # Build DataFrame with CORRECT column names
        all_cols = list(X.columns) + [y_col]
        df_synth = pd.DataFrame(syn_data, columns=all_cols)

        # Split back to X/y using original names
        x_synth = df_synth[X.columns]
        y_synth = df_synth[[y_col]]   # keep as DataFrame so header is saved

        #  Now synthetic X has SAME feature names as real X
        x_synth.to_csv(os.path.join(save_dir, "x_synth.csv"), index=False)
        y_synth.to_csv(os.path.join(save_dir, "y_synth.csv"),
                       index=False, header=True)

        print(f"\nSynthetic data saved to: {save_dir}")


#  MAGIC dataset setup 
dataset_name = "magic"

raw_csv = ROOT / "raw_data" / f"{dataset_name}.csv"
disc_csv = ROOT / "discretized_data" / f"{dataset_name}.csv"
disc_csv.parent.mkdir(parents=True, exist_ok=True)

print(f"Discretizing {raw_csv} -> {disc_csv}")
discretize_preprocess(str(raw_csv), str(disc_csv))

input_csv     = str(disc_csv)
output_dir    = str(ROOT / "sample_data" / dataset_name)
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / dataset_name / "ganblr")

print("input_csv    :", input_csv)
print("output_dir   :", output_dir)
print("real_test_dir:", real_test_dir)
print("synthetic_dir:", synthetic_dir)

#  Run pipeline with CustomGANBLR
pipeline = TrainTestSplitPipeline(model=CustomGANBLR)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    size_category="medium",   # MAGIC = medium
)

print("\nTSTR Evaluation Results (GANBLR on MAGIC):")
print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\magic.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\magic.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
input_csv    : C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
output_dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\magic
real_test_dir: C:\Users\Prabu\Downloads\Katabatic\sample_data\magic
synthetic_dir: C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\ganblr
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
0    0.648396
1    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.648265
1    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (

Generating for node: 4: 100%|██████████| 11/11 [00:00<00:00, 27.56it/s]



Synthetic data saved to: C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\ganblr


C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results\magic\ganblr_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7713
F1 Score: 0.7742
AUC: 0.8381

MLP:
Accuracy: 0.7358
F1 Score: 0.6804
AUC: 0.8583

RF:
Accuracy: 0.7400
F1 Score: 0.6870
AUC: 0.8638

XGBoost:
Accuracy: 0.7497
F1 Score: 0.7028
AUC: 0.8486

TSTR Evaluation Results (GANBLR on MAGIC):
Train test split pipeline executed successfully.
